In [1]:
import sys
import os

# Add the parent directory to Python's search path
sys.path.append(os.path.abspath('..'))

In [2]:
import json
from quantum_qr.evaluate import evaluate_corpus

fixtures_dir = "../data/fixtures/"
manifest_path = "../data/fixtures/manifest.json"

print("Evaluating corpus on AerSimulator (Noiseless Baseline)...")
report = evaluate_corpus(fixtures_dir, manifest_path)

metrics = report["metrics"]
cm = report["confusion_matrix"]

print("\n--- Evaluation Summary ---")
print(f"Total Fixtures: {metrics['total']}")
print(f"Accuracy:       {metrics['accuracy'] * 100:.1f}%")
print(f"Recall:         {metrics['recall'] * 100:.1f}%")
print(f"Precision:      {metrics['precision'] * 100:.1f}%")

print("\n--- Confusion Matrix ---")
print(f"True Positives (Tampered Caught):   {cm['TP']}")
print(f"True Negatives (Authentic Passed):  {cm['TN']}")
print(f"False Positives (False Alarms):     {cm['FP']}")
print(f"False Negatives (Forgeries Missed): {cm['FN']}")

print("\n--- Tamper Breakdown ---")
for t_type, stats in report["breakdown"].items():
    print(f"{t_type.capitalize()}: {stats['caught']}/{stats['total']} caught "
          f"({stats['recall'] * 100:.1f}% Recall)")

Evaluating corpus on AerSimulator (Noiseless Baseline)...

--- Evaluation Summary ---
Total Fixtures: 5
Accuracy:       100.0%
Recall:         100.0%
Precision:      100.0%

--- Confusion Matrix ---
True Positives (Tampered Caught):   4
True Negatives (Authentic Passed):  1
False Positives (False Alarms):     0
False Negatives (Forgeries Missed): 0

--- Tamper Breakdown ---
Data_tampered: 1/1 caught (100.0% Recall)
Nonce_tampered: 1/1 caught (100.0% Recall)
Tag_tampered: 1/1 caught (100.0% Recall)
Forged: 1/1 caught (100.0% Recall)


### Interpreting the Baseline: Why 100% is Expected

On this run, the verifier achieved a 100% Recall and 100% Accuracy score. **This is expected, not impressive.** Running this pipeline on Qiskit's noiseless `AerSimulator` confirms three critical engineering milestones:
1. **Corpus Consistency:** The fixtures and their expected tags were generated correctly.
2. **Pipeline Integration:** The classical cryptography (HMAC) seamlessly hands off to the quantum circuit generation.
3. **Endianness Alignment:** Qiskit's little-endian measurement outputs are correctly reversed and mapped back to the classical secret strings.

However, a perfect score on a simulator is *not* evidence that quantum computing adds detection power. On a perfect, noiseless simulator, the Deutsch-Jozsa readout matches the classical secret via phase interference by absolute mathematical construction. It cannot fail.

The meaningful accuracy measurement happens later in this project (Days 17–18) when this pipeline is deployed on **noisy quantum hardware** (IBM Quantum). There, this exact evaluation harness will be reused to quantify how hardware noise, gate errors, and decoherence degrade detection reliability. 

Today's deliverable is the measurement framework itself, with the simulator result serving as a clean, verified baseline.

In [3]:
import json

# 1. Ensure the output directory exists
os.makedirs("data", exist_ok=True)

# 2. Re-run the evaluation (or reuse the 'report' variable from Task 4)
# We ensure the results are saved as a structured JSON file
output_path = "../data/eval_simulator.json"

print(f"Saving baseline results to {output_path}...")

with open(output_path, 'w') as f:
    json.dump(report, f, indent=4)

print("✅ Baseline frozen. You now have a verifiable simulator benchmark.")

Saving baseline results to ../data/eval_simulator.json...
✅ Baseline frozen. You now have a verifiable simulator benchmark.
